In [1]:
from lm_polygraph.utils import UEManager 



man = UEManager.load('coqa_gpt40_mini.man')

man.estimations.keys()


/home/maiya.goloburda/.conda/envs/kernels/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/maiya.goloburda/.conda/envs/kernels/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


dict_keys([('sequence', 'LexicalSimilarity_rouge1'), ('sequence', 'LexicalSimilarity_rouge2'), ('sequence', 'LexicalSimilarity_rougeL'), ('sequence', 'LexicalSimilarity_BLEU'), ('sequence', 'NumSemSets'), ('sequence', 'EigValLaplacian_NLI_score_entail'), ('sequence', 'EigValLaplacian_NLI_score_contra'), ('sequence', 'EigValLaplacian_Jaccard_score'), ('sequence', 'DegMat_NLI_score_entail'), ('sequence', 'DegMat_NLI_score_contra'), ('sequence', 'DegMat_Jaccard_score'), ('sequence', 'Eccentricity_NLI_score_entail'), ('sequence', 'Eccentricity_NLI_score_contra'), ('sequence', 'Eccentricity_Jaccard_score'), ('sequence', 'SemanticEntropyEmpirical'), ('sequence', 'PTrueEmpirical'), ('sequence', 'LabelProb'), ('sequence', 'CocoaConsistency'), ('sequence', 'GreedyKernelScore')])

In [2]:
import json

file_path = "prompt_runs_ckpt/ckpt_confidence.jsonl"
records_confidence = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records_confidence.append(json.loads(line))



file_path = "prompt_runs_ckpt/ckpt_reflect_prob.jsonl"
records_source = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records_source.append(json.loads(line))

            


In [3]:
# records_confidence[0]


verbalized_2s = [1-x['confidence_only'] for x in records_confidence]

verbalized_source =[1-x['reflect_prob'] for x in records_source]




In [4]:
import numpy as np

man.estimations[('sequence', 'Verbalized_2s')] = np.array(verbalized_2s)

man.estimations[('sequence', 'Verbalied_Source')] = np.array(verbalized_source)


from lm_polygraph.ue_metrics import PredictionRejectionArea

ue_metrics = [PredictionRejectionArea(),PredictionRejectionArea(0.5)]

man.ue_metrics=ue_metrics

man.eval_ue()



We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 nans in Rouge_rouge2 generation metric.
We got 1005 

In [5]:
methods = {key[1] for key in man.metrics.keys()}


print(methods)

results = []
for method in methods:
    prr_score = man.metrics[('sequence', method, 'AlignScore', 'prr_0.5_normalized')]
    results.append({'method': method, 'prr': prr_score})
import pandas as pd 

# create DataFrame and sort by performance
df = pd.DataFrame(results).sort_values(by="prr", ascending=False).reset_index(drop=True)

print(df)



{'LexicalSimilarity_BLEU', 'Eccentricity_NLI_score_entail', 'PTrueEmpirical', 'LexicalSimilarity_rouge2', 'NumSemSets', 'Eccentricity_NLI_score_contra', 'Verbalized_2s', 'CocoaConsistency', 'DegMat_NLI_score_contra', 'DegMat_NLI_score_entail', 'LexicalSimilarity_rougeL', 'LexicalSimilarity_rouge1', 'EigValLaplacian_Jaccard_score', 'EigValLaplacian_NLI_score_entail', 'Verbalied_Source', 'DegMat_Jaccard_score', 'LabelProb', 'EigValLaplacian_NLI_score_contra', 'GreedyKernelScore', 'SemanticEntropyEmpirical', 'Eccentricity_Jaccard_score'}
                              method       prr
0      Eccentricity_NLI_score_contra  0.364013
1      Eccentricity_NLI_score_entail  0.359654
2   EigValLaplacian_NLI_score_contra  0.352926
3           LexicalSimilarity_rougeL  0.340773
4      EigValLaplacian_Jaccard_score  0.340316
5           LexicalSimilarity_rouge1  0.339923
6           LexicalSimilarity_rouge2  0.339641
7   EigValLaplacian_NLI_score_entail  0.338007
8               DegMat_Jaccard_score

In [6]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ---------------------------
# 1) HOW TO FETCH YOUR DATA
# ---------------------------
# TODO: Adapt these two getters to your setup.
# Each must return a 1D numpy array (same length) over *examples* for a given method.
GETTERS = {
    "alignscore": lambda method: np.array(man.gen_metrics[('sequence', 'AlignScore')]),
    "confidence": lambda method: 1- np.array(man.estimations[('sequence', method)]),
    
}
# If you don't have `man.get_per_example`, replace with whatever you use, e.g.:
# GETTERS = {
#     "alignscore": lambda method: df[df.method==method]["align_score"].to_numpy(),
#     "confidence": lambda method: df[df.method==method]["conf"].to_numpy(),
# }

# ---------------------------
# 2) METRIC HELPERS
# ---------------------------
def safe_spearman(a, b):
    if len(a) < 2 or np.all(a == a[0]) or np.all(b == b[0]):
        return np.nan
    return stats.spearmanr(a, b).correlation

def safe_pearson(a, b):
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return stats.pearsonr(a, b)[0]

def ece(conf, target, n_bins=15):
    """Expected Calibration Error for continuous target in [0,1].
    We bin by predicted confidence; within each bin, compare mean(conf) vs mean(target).
    Uses right=True so conf==1.0 lands in the last bin."""
    conf = np.asarray(conf, dtype=float)
    target = np.asarray(target, dtype=float)

    # guard / clip
    conf = np.clip(conf, 0.0, 1.0)
    target = np.clip(target, 0.0, 1.0)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    # right=True ensures values equal to a bin edge go to the (left) bin;
    # specifically conf==1.0 will map to the final bin index.
    inds = np.digitize(conf, bins, right=True) - 1  # -> [0..n_bins-1]

    ece_val, total = 0.0, len(conf)
    for b in range(n_bins):
        mask = (inds == b)
        if not np.any(mask):
            continue
        gap = np.abs(conf[mask].mean() - target[mask].mean())
        ece_val += (mask.sum() / total) * gap
    return float(ece_val)

def to_binary(y_cont, thresh=0.8):
    """If you want discriminative metrics (AUC/AP), binarize AlignScore."""
    return (np.asarray(y_cont) >= thresh).astype(int)

# ---------------------------
# 3) COLLECT METHODS & COMPUTE
# ---------------------------
# Unique method names (adjust index if your tuple layout differs)
methods = sorted({k[1] for k in man.metrics.keys()})

rows = []
for method in methods:
    try:
        y_true = GETTERS["alignscore"](method).astype(float)   # continuous reference in [0,1]
        y_conf = GETTERS["confidence"](method).astype(float)   # model confidence in [0,1]
    except Exception as e:
        print(f"[WARN] Skipping {method}: failed to fetch arrays ({e})")
        continue

    # basic sanity
    m = min(len(y_true), len(y_conf))
    y_true, y_conf = y_true[:m], y_conf[:m]
    mask = np.isfinite(y_true) & np.isfinite(y_conf)
    y_true, y_conf = y_true[mask], y_conf[mask]
    if len(y_true) == 0:
        print(f"[WARN] Skipping {method}: no valid pairs")
        continue

    # clip to [0,1]
    y_true = np.clip(y_true, 0, 1)
    y_conf = np.clip(y_conf, 0, 1)

    # association
    pear = safe_pearson(y_true, y_conf)
    spear = safe_spearman(y_true, y_conf)

    # calibration-like
    ece15 = ece(y_conf, y_true, n_bins=15)  # <-- added

    # discriminative (treat AlignScore >= 0.5 as "good")
    y_bin = to_binary(y_true, thresh=0.5)
    auc = roc_auc_score(y_bin, y_conf) if len(np.unique(y_bin)) > 1 else np.nan
    ap = average_precision_score(y_bin, y_conf) if len(np.unique(y_bin)) > 1 else np.nan

    # regression error (MSE equals Brier when y_true is binary)
    mse = float(np.mean((y_true - y_conf) ** 2))

    rows.append({
        "method": method,
        "n": len(y_true),
        "pearson": pear,
        "spearman": spear,
        "mse": mse,
        "ece@15": ece15,        # <-- added
        "auc(align>=0.5)": auc,
        "ap(align>=0.5)": ap,
    })

# ---------------------------
# 4) SHOW RESULTS
# ---------------------------
df = pd.DataFrame(rows)
# Sort by what you care about; here we sort by Spearman
df_sorted = df.sort_values(by=["spearman"], ascending=[False]).reset_index(drop=True)

# Nicely formatted print
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 120):
    print(df_sorted.round({
        "pearson": 3, "spearman": 3,
        "mse": 4, "ece@15": 4,          # <-- added
        "auc(align>=0.5)": 3, "ap(align>=0.5)": 3
    }))


                              method     n  pearson  spearman     mse  ece@15  auc(align>=0.5)  ap(align>=0.5)
0               DegMat_Jaccard_score  2000    0.318     0.323  0.1863  0.1881            0.669           0.818
1           SemanticEntropyEmpirical  2000    0.313     0.315  0.1919  0.1792            0.644           0.801
2                          LabelProb  2000    0.297     0.313  0.1922  0.1964            0.643           0.800
3                   CocoaConsistency  2000    0.291     0.313  0.1882  0.1872            0.605           0.761
4         Eccentricity_Jaccard_score  2000    0.297     0.312  0.2632  0.1714            0.661           0.811
5      Eccentricity_NLI_score_entail  2000    0.309     0.300  0.2404  0.1735            0.678           0.821
6      Eccentricity_NLI_score_contra  2000    0.253     0.295  0.2188  0.2253            0.687           0.827
7                   Verbalied_Source  2000    0.220     0.247  0.2015  0.2043            0.628           0.796
8

In [10]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import LeaveOneOut, KFold
from lm_polygraph.normalizers.isotonic_pcc import IsotonicPCCNormalizer

# ---------------------------
# 1) HOW TO FETCH YOUR DATA
# ---------------------------
GETTERS = {
    "alignscore": lambda method: np.array(man.gen_metrics[('sequence', 'AlignScore')]),
    "confidence": lambda method: 1 - np.array(man.estimations[('sequence', method)]),
}

# ---------------------------
# 2) METRIC HELPERS
# ---------------------------
def safe_spearman(a, b):
    if len(a) < 2 or np.all(a == a[0]) or np.all(b == b[0]):
        return np.nan
    return stats.spearmanr(a, b).correlation

def safe_pearson(a, b):
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return stats.pearsonr(a, b)[0]

def ece(conf, target, n_bins=15):
    """ECE for continuous target in [0,1]. Bin by conf, compare means."""
    conf = np.asarray(conf, dtype=float)
    target = np.asarray(target, dtype=float)
    conf = np.clip(conf, 0.0, 1.0)
    target = np.clip(target, 0.0, 1.0)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    inds = np.digitize(conf, bins, right=True) - 1  # [0..n_bins-1]

    ece_val, total = 0.0, len(conf)
    for b in range(n_bins):
        mask = (inds == b)
        if not np.any(mask):
            continue
        gap = np.abs(conf[mask].mean() - target[mask].mean())
        ece_val += (mask.sum() / total) * gap
    return float(ece_val)

def to_binary(y_cont, thresh=0.8):
    return (np.asarray(y_cont) >= thresh).astype(int)

# --- NEW: isotonic calibration error helper -----------------
def iso_cal_error(est, y, cv="loo", n_splits=5, loss="mse", use_pcc=True):
    """
    Out-of-fold isotonic calibration error between estimator (UE) and target.
    - cv: 'loo' or 'kfold'
    - loss: 'mse' or 'mae'
    - use_pcc: if True, use IsotonicPCCNormalizer (your class); else sklearn IsotonicRegression
    """
    est = np.asarray(est, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(est) & np.isfinite(y)
    est, y = est[m], y[m]
    n = len(y)
    if n == 0:
        return np.nan

    # Degenerate safeguards
    if np.all(est == est[0]) or np.all(y == y[0]):
        yhat = np.full(n, np.nanmean(y))
        return float(np.mean((yhat - y) ** 2)) if loss == "mse" else float(np.mean(np.abs(yhat - y)))

    splitter = LeaveOneOut() if cv == "loo" else KFold(n_splits=min(n_splits, n), shuffle=True, random_state=0)
    yhat = np.empty(n, dtype=float)

    for tr_idx, te_idx in splitter.split(est):
        x_tr, y_tr = est[tr_idx], y[tr_idx]
        x_te = est[te_idx]

        if use_pcc:
            # Your normalizer: API in your snippet was fit(target, estimator), transform(estimator)
            normalizer = IsotonicPCCNormalizer()
            normalizer.fit(y_tr, x_tr)
            yhat[te_idx] = normalizer.transform(np.asarray(x_te))
        else:
            # Fallback to sklearn isotonic (monotone map est->y), clip for OOB
            iso = IsotonicRegression(increasing="auto", out_of_bounds="clip")
            iso.fit(x_tr, y_tr)
            yhat[te_idx] = iso.predict(x_te)

    if loss == "mse":
        return float(np.mean((yhat - y) ** 2))
    else:
        return float(np.mean(np.abs(yhat - y)))

# ---------------------------
# 3) COLLECT METHODS & COMPUTE
# ---------------------------
methods = sorted({k[1] for k in man.metrics.keys()})

rows = []
for method in methods:
    try:
        y_true = GETTERS["alignscore"](method).astype(float)   # continuous reference (any range OK)
        y_conf = GETTERS["confidence"](method).astype(float)   # UE/estimator (any range OK)
    except Exception as e:
        print(f"[WARN] Skipping {method}: failed to fetch arrays ({e})")
        continue

    # align to same length & finite
    m = min(len(y_true), len(y_conf))
    y_true, y_conf = y_true[:m], y_conf[:m]
    mask = np.isfinite(y_true) & np.isfinite(y_conf)
    y_true, y_conf = y_true[mask], y_conf[mask]
    if len(y_true) == 0:
        print(f"[WARN] Skipping {method}: no valid pairs")
        continue

    # For correlation and ECE we’ll clip to [0,1] copies (non-destructive)
    y_true_01 = np.clip(y_true, 0, 1)
    y_conf_01 = np.clip(y_conf, 0, 1)

    # association
    pear = safe_pearson(y_true, y_conf)
    spear = safe_spearman(y_true, y_conf)

    # calibration-like
    ece15 = ece(y_conf_01, y_true_01, n_bins=15)

    # discriminative (treat AlignScore >= 0.5 as "good")
    y_bin = to_binary(y_true_01, thresh=0.5)
    auc = roc_auc_score(y_bin, y_conf_01) if len(np.unique(y_bin)) > 1 else np.nan
    ap = average_precision_score(y_bin, y_conf_01) if len(np.unique(y_bin)) > 1 else np.nan

    # regression errors
    mse = float(np.mean((y_true_01 - y_conf_01) ** 2))  # on [0,1] scale for comparability

    # NEW: out-of-fold isotonic calibrated error (your PCC normalizer by default)
    iso_mse_loo = iso_cal_error(y_conf, y_true, cv="loo",  loss="mse", use_pcc=True)
    iso_mae_k5  = iso_cal_error(y_conf, y_true, cv="kfold", n_splits=5, loss="mae", use_pcc=True)

    rows.append({
        "method": method,
        "n": len(y_true),
        "pearson": pear,
        "spearman": spear,
        "mse": mse,
        "ece@15": ece15,
        "auc(align>=0.5)": auc,
        "ap(align>=0.5)": ap,
        "iso-cal-mse(loo)": iso_mse_loo,   # NEW
        "iso-cal-mae(k=5)": iso_mae_k5,    # NEW
    })

# ---------------------------
# 4) SHOW RESULTS
# ---------------------------
df = pd.DataFrame(rows)
df_sorted = df.sort_values(by=["iso-cal-mse(loo)", "spearman"], ascending=[True, False]).reset_index(drop=True)

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 160):
    print(df_sorted.round({
        "pearson": 3, "spearman": 3,
        "mse": 4, "ece@15": 4,
        "auc(align>=0.5)": 3, "ap(align>=0.5)": 3,
        "iso-cal-mse(loo)": 6, "iso-cal-mae(k=5)": 6,
    }))


                              method     n  pearson  spearman     mse  ece@15  auc(align>=0.5)  ap(align>=0.5)  iso-cal-mse(loo)  iso-cal-mae(k=5)
0           SemanticEntropyEmpirical  2000    0.569     0.379  0.1067  0.1008            0.793           0.907          0.126147          0.288210
1            DegMat_NLI_score_entail  2000    0.584     0.378  0.0856  0.0576            0.793           0.904          0.126147          0.288210
2                          LabelProb  2000    0.560     0.377  0.0949  0.0925            0.790           0.906          0.126147          0.288210
3                         NumSemSets  2000    0.537     0.377  0.7748  0.0000            0.500           0.792          0.126147          0.288210
4               DegMat_Jaccard_score  2000    0.579     0.362  0.0913  0.0835            0.803           0.914          0.126147          0.288210
5           LexicalSimilarity_rougeL  2000    0.579     0.361  0.1639  0.1945            0.500           0.792        

In [1]:
from lm_polygraph.utils import UEManager 



man = UEManager.load('coqa_gpt40_mini.man')

man.estimations.keys()


/home/maiya.goloburda/.conda/envs/kernels/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/maiya.goloburda/.conda/envs/kernels/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


dict_keys([('sequence', 'LexicalSimilarity_rouge1'), ('sequence', 'LexicalSimilarity_rouge2'), ('sequence', 'LexicalSimilarity_rougeL'), ('sequence', 'LexicalSimilarity_BLEU'), ('sequence', 'NumSemSets'), ('sequence', 'EigValLaplacian_NLI_score_entail'), ('sequence', 'EigValLaplacian_NLI_score_contra'), ('sequence', 'EigValLaplacian_Jaccard_score'), ('sequence', 'DegMat_NLI_score_entail'), ('sequence', 'DegMat_NLI_score_contra'), ('sequence', 'DegMat_Jaccard_score'), ('sequence', 'Eccentricity_NLI_score_entail'), ('sequence', 'Eccentricity_NLI_score_contra'), ('sequence', 'Eccentricity_Jaccard_score'), ('sequence', 'SemanticEntropyEmpirical'), ('sequence', 'PTrueEmpirical'), ('sequence', 'LabelProb'), ('sequence', 'CocoaConsistency'), ('sequence', 'GreedyKernelScore')])

In [3]:
man.stats['input_texts'][0]



"Here's a short story:\n\nWhen my father was dying, I traveled a thousand miles from home to be with him in his last days. It was far more heartbreaking than I'd expected, one of the most difficult and painful times in my life. After he passed away I stayed alone in his apartment. There were so many things to deal with. It all seemed endless. I was lonely. I hated the silence of the apartment. \n\nBut one evening the silence was broken: I heard crying outside. I opened the door to find a little cat on the steps. He was thin and poor. He looked the way I felt. I brought him inside and gave him a can of fish. He ate it and then almost immediately fell sound asleep. \n\nThe next morning I checked with neighbors and learned that the cat had been abandoned by his owner who's moved out. So the little cat was there all alone, just like I was. As I walked back to the apartment, I tried to figure out what to do with him. Having something else to take care of seemed _ But as soon as I opened the